In [33]:
!pip install xgboost joblib -q

In [34]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

In [35]:
df = pd.read_csv("model2_waiting_time_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Shape: (5779, 18)

Columns:
['station_id', 'location_type', 'vehicle_type', 'battery_capacity_kWh', 'initial_soc', 'charging_power_kW', 'queue_length', 'station_load', 'electricity_price', 'renewable_energy_ratio', 'traffic_density', 'weather_condition', 'day_of_week', 'time_slot', 'charging_demand', 'charging_priority', 'arrival_hour', 'waiting_time']


,station_id,location_type,vehicle_type,battery_capacity_kWh,initial_soc,charging_power_kW,queue_length,station_load,electricity_price,renewable_energy_ratio,traffic_density,weather_condition,day_of_week,time_slot,charging_demand,charging_priority,arrival_hour,waiting_time
0,ST004,Urban,Two-Wheeler,60,48.984550,7,4,15.161672,13.66,0.280335,Low,Cloudy,Wednesday,Off-Peak,17.242398,Low,0,21.120
1,ST005,Urban,Two-Wheeler,100,58.495493,50,3,20.997219,5.47,0.392127,Low,Rainy,Wednesday,Off-Peak,18.324933,Low,0,7.663
2,ST019,Highway,Car,75,35.711722,50,8,31.606151,9.50,0.103979,Low,Clear,Wednesday,Off-Peak,36.028168,Low,0,17.842
3,ST008,Urban,Two-Wheeler,40,29.270825,11,3,21.803050,6.22,0.248553,Low,Clear,Wednesday,Off-Peak,17.146935,Medium,0,17.318
4,ST008,Highway,Two-Wheeler,75,25.585554,11,5,15.626266,13.42,0.234926,Low,Cloudy,Wednesday,Off-Peak,14.577768,Low,1,18.741


In [36]:
print("Missing values:")
print(df.isnull().sum())

print("\nTarget statistics:")
print(df["waiting_time"].describe())

print("\nTarget sample:")
print(df["waiting_time"].head(10))

Missing values:
station_id                0
location_type             0
vehicle_type              0
battery_capacity_kWh      0
initial_soc               0
charging_power_kW         0
queue_length              0
station_load              0
electricity_price         0
renewable_energy_ratio    0
traffic_density           0
weather_condition         0
day_of_week               0
time_slot                 0
charging_demand           0
charging_priority         0
arrival_hour              0
waiting_time              0
dtype: int64

Target statistics:
count    5779.000000
mean       18.844122
std         6.931409
min         0.100000
25%        13.978500
50%        18.961000
75%        23.738000
max        40.240000
Name: waiting_time, dtype: float64

Target sample:
0    21.120
1     7.663
2    17.842
3    17.318
4    18.741
5     2.092
6     4.184
7    10.082
8    10.237
9    11.181
Name: waiting_time, dtype: float64


In [37]:
target = "waiting_time"

X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

X shape: (5779, 17)
y shape: (5779,)

Features:
['station_id', 'location_type', 'vehicle_type', 'battery_capacity_kWh', 'initial_soc', 'charging_power_kW', 'queue_length', 'station_load', 'electricity_price', 'renewable_energy_ratio', 'traffic_density', 'weather_condition', 'day_of_week', 'time_slot', 'charging_demand', 'charging_priority', 'arrival_hour']


In [38]:
numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['battery_capacity_kWh', 'initial_soc', 'charging_power_kW', 'queue_length', 'station_load', 'electricity_price', 'renewable_energy_ratio', 'charging_demand', 'arrival_hour']

Categorical features:
['station_id', 'location_type', 'vehicle_type', 'traffic_density', 'weather_condition', 'day_of_week', 'time_slot', 'charging_priority']


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 4623
Testing rows: 1156


In [40]:
numeric_features = [
    "battery_capacity_kWh",
    "initial_soc",
    "charging_power_kW",
    "queue_length",
    "station_load",
    "electricity_price",
    "renewable_energy_ratio",
    "charging_demand",
    "arrival_hour"
]

categorical_features = [
    "station_id",
    "location_type",
    "vehicle_type",
    "traffic_density",
    "weather_condition",
    "day_of_week",
    "time_slot",
    "charging_priority"
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing complete!")
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Preprocessing complete!
Training shape: (4623, 52)
Testing shape: (1156, 52)


In [41]:
baseline_prediction = np.full(
    len(y_test),
    y_train.mean()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

baseline_mape = mean_absolute_percentage_error(
    y_test,
    baseline_prediction
) * 100

print("BASELINE")
print("MAE :", round(baseline_mae, 3), "minutes")
print("RMSE:", round(baseline_rmse, 3), "minutes")
print("R²  :", round(baseline_r2, 3))
print("MAPE:", round(baseline_mape, 2), "%")

BASELINE
MAE : 5.58 minutes
RMSE: 6.867 minutes
R²  : -0.002
MAPE: 45.91 %


In [42]:
rf_wait = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_wait.fit(
    X_train_processed,
    y_train
)

rf_wait_predictions = rf_wait.predict(
    X_test_processed
)

rf_wait_mae = mean_absolute_error(
    y_test,
    rf_wait_predictions
)

rf_wait_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_wait_predictions
    )
)

rf_wait_r2 = r2_score(
    y_test,
    rf_wait_predictions
)

rf_wait_mape = mean_absolute_percentage_error(
    y_test,
    rf_wait_predictions
) * 100

print("RANDOM FOREST — MODEL 2")
print("MAE :", round(rf_wait_mae, 3), "minutes")
print("RMSE:", round(rf_wait_rmse, 3), "minutes")
print("R²  :", round(rf_wait_r2, 3))
print("MAPE:", round(rf_wait_mape, 2), "%")

RANDOM FOREST — MODEL 2
MAE : 1.613 minutes
RMSE: 2.06 minutes
R²  : 0.91
MAPE: 11.51 %


In [43]:
xgb_wait = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb_wait.fit(
    X_train_processed,
    y_train
)

xgb_wait_predictions = xgb_wait.predict(
    X_test_processed
)

xgb_wait_mae = mean_absolute_error(
    y_test,
    xgb_wait_predictions
)

xgb_wait_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_wait_predictions
    )
)

xgb_wait_r2 = r2_score(
    y_test,
    xgb_wait_predictions
)

xgb_wait_mape = mean_absolute_percentage_error(
    y_test,
    xgb_wait_predictions
) * 100

print("XGBOOST — MODEL 2")
print("MAE :", round(xgb_wait_mae, 3), "minutes")
print("RMSE:", round(xgb_wait_rmse, 3), "minutes")
print("R²  :", round(xgb_wait_r2, 3))
print("MAPE:", round(xgb_wait_mape, 2), "%")

XGBOOST — MODEL 2
MAE : 1.37 minutes
RMSE: 1.74 minutes
R²  : 0.936
MAPE: 9.6 %


In [44]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        baseline_mae,
        rf_wait_mae,
        xgb_wait_mae
    ],
    "RMSE": [
        baseline_rmse,
        rf_wait_rmse,
        xgb_wait_rmse
    ],
    "R2": [
        baseline_r2,
        rf_wait_r2,
        xgb_wait_r2
    ],
    "MAPE": [
        baseline_mape,
        rf_wait_mape,
        xgb_wait_mape
    ]
})

print(results.round(3).to_string(index=False))

        Model   MAE  RMSE     R2   MAPE
     Baseline 5.580 6.867 -0.002 45.912
Random Forest 1.613 2.060  0.910 11.510
      XGBoost 1.370 1.740  0.936  9.599


In [45]:
waiting_time_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_wait)
])

joblib.dump(
    waiting_time_pipeline,
    "/content/waiting_time_model.pkl"
)

print("Model 2 saved:")
print("/content/waiting_time_model.pkl")

# Verify the saved model works
loaded_model = joblib.load(
    "/content/waiting_time_model.pkl"
)

test_prediction = loaded_model.predict(
    pd.DataFrame([example_input])
)[0]

print(
    " Test prediction:",
    round(float(test_prediction), 2),
    "minutes"
)

Model 2 saved:
/content/waiting_time_model.pkl
 Test prediction: 9.44 minutes


In [46]:

def predict_wait_time(input_data):

    input_df = pd.DataFrame([input_data])

    if "arrival_hour" not in input_df.columns:
        raise ValueError("arrival_hour is required")

    prediction = loaded_model.predict(input_df)[0]

    q25 = df["queue_length"].quantile(0.25)
    q50 = df["queue_length"].quantile(0.50)
    q75 = df["queue_length"].quantile(0.75)

    queue_value = float(input_df["queue_length"].iloc[0])

    if queue_value <= q25:
        queue_level = "Low"
    elif queue_value <= q50:
        queue_level = "Moderate"
    elif queue_value <= q75:
        queue_level = "High"
    else:
        queue_level = "Critical"

    load_value = float(input_df["station_load"].iloc[0])

    load_q25 = df["station_load"].quantile(0.25)
    load_q50 = df["station_load"].quantile(0.50)
    load_q75 = df["station_load"].quantile(0.75)

    if load_value <= load_q25:
        congestion_level = "Low"
    elif load_value <= load_q50:
        congestion_level = "Moderate"
    elif load_value <= load_q75:
        congestion_level = "High"
    else:
        congestion_level = "Critical"

    return {
        "station_id": str(input_df["station_id"].iloc[0]),
        "time_period": f"{int(input_df['arrival_hour'].iloc[0]):02d}:00",
        "predicted_wait_time_minutes": round(float(prediction), 2),
        "queue_level": queue_level,
        "congestion_level": congestion_level,
        "model_error": {
            "mae_minutes": round(float(xgb_wait_mae), 3),
            "rmse_minutes": round(float(xgb_wait_rmse), 3)
        }
    }

print("Backend prediction function created!")

Backend prediction function created!


In [51]:
test_row = X_test.iloc[0].copy()

example_input = test_row.to_dict()

prediction_output = predict_wait_time(example_input)

print(json.dumps(
    prediction_output,
    indent=4
))

{
    "station_id": "ST004",
    "time_period": "15:00",
    "predicted_wait_time_minutes": 9.44,
    "queue_level": "Low",
    "congestion_level": "Moderate",
    "model_error": {
        "mae_minutes": 1.37,
        "rmse_minutes": 1.74
    }
}


In [52]:
with open(
    "/content/model_2_sample_output.json",
    "w"
) as f:
    json.dump(
        prediction_output,
        f,
        indent=4
    )

print("model_2_sample_output.json created")

model_2_sample_output.json created


In [54]:

model_2_metadata = {
    "model_name": "EV Charging Wait-Time Prediction",
    "model_version": "1.0",
    "model_type": "XGBoost Regressor",
    "model_file": "waiting_time_model.pkl",

    "target": "waiting_time",
    "target_unit": "minutes",

    "input_features": [
        "station_id",
        "location_type",
        "vehicle_type",
        "battery_capacity_kWh",
        "initial_soc",
        "charging_power_kW",
        "queue_length",
        "station_load",
        "electricity_price",
        "renewable_energy_ratio",
        "traffic_density",
        "weather_condition",
        "day_of_week",
        "time_slot",
        "charging_demand",
        "charging_priority",
        "arrival_hour"
    ],

    "outputs": [
        "station_id",
        "time_period",
        "predicted_wait_time_minutes",
        "queue_level",
        "congestion_level",
        "model_error"
    ],

    "evaluation_metrics": {
        "MAE_minutes": round(float(xgb_wait_mae), 3),
        "RMSE_minutes": round(float(xgb_wait_rmse), 3),
        "R2": round(float(xgb_wait_r2), 3),
        "MAPE_percent": round(float(xgb_wait_mape), 3)
    },

    "reliability": {
        "MAE_minutes": round(float(xgb_wait_mae), 3),
        "RMSE_minutes": round(float(xgb_wait_rmse), 3),
        "note": "MAE and RMSE represent prediction error on the held-out test set. They are not a confidence percentage."
    },

    "data_note": "The waiting_time target was synthetically constructed from the EV charging dataset. Therefore, evaluation metrics describe performance on the synthetic target and should not be interpreted as measured real-world waiting-time accuracy."
}

with open("/content/model_2_metadata.json", "w") as f:
    json.dump(model_2_metadata, f, indent=4)

print("model_2_metadata.json updated")

model_2_metadata.json updated


In [53]:
test_input = {
    key: (
        value.item()
        if hasattr(value, "item")
        else value
    )
    for key, value in example_input.items()
}

with open(
    "/content/model_2_test_input.json",
    "w"
) as f:
    json.dump(
        test_input,
        f,
        indent=4
    )

print(" model_2_test_input.json created")

 model_2_test_input.json created
